# Shinhan Card 파일 불러오기 (정산월 날짜 보정 적용)

## 0. 초기 세팅

- 신한카드 사이트에서, 이용대금명세서를 받으면 XLS로 저장되는데 양식은 HTML형식임

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent
if str(project_root / "src") not in sys.path:
    sys.path.append(str(project_root / "src"))

import pandas as pd
import numpy as np
from ledgerly.expenditure import (
    shinhan_file_config,
    load_shinhan_html_file,
    preprocess_shinhan_data,
    map_shinhan_card_df_to_expenditure,
    map_category,
    insert_expenditure_data
)

## 1. 데이터 파일 조회 및 정밀 파싱

In [ ]:
target_yymm = "2603"
target_filename = "shinhancard_2603.xls"
target_file = shinhan_file_config["data_dir"] / target_yymm / target_filename

print(f"Target file: {target_file}")

raw_df = load_shinhan_html_file(target_file)

if not raw_df.empty:
    sh_df = preprocess_shinhan_data(raw_df)
    print(f"총 {len(sh_df)}건의 데이터를 추출했습니다.")
    display(sh_df.head())
else:
    print("데이터를 찾지 못했습니다.")
    sh_df = pd.DataFrame()

## 2. DB 데이터 매핑 및 중복 제거
정산 월 정보를 전달하여 과거 할부 내역 등의 거래일자를 보정합니다.

In [ ]:
if not sh_df.empty:
    # 정산 기준일 설정 (예: 2026-03-01)
    statement_date = f"20{target_yymm[:2]}-{target_yymm[2:]}-01"
    
    expenditure_df = map_shinhan_card_df_to_expenditure(sh_df, statement_date=statement_date)
    
    # source_uid 기준 중복 제거
    before_count = len(expenditure_df)
    expenditure_df = expenditure_df.drop_duplicates(subset=["source_uid"])
    after_count = len(expenditure_df)
    
    if before_count != after_count:
        print(f"중복 데이터 {before_count - after_count}건을 제거하였습니다.")
    
    # 유효 데이터 필터링 및 카테고리 매핑
    expenditure_df = expenditure_df[expenditure_df["amount"] > 0]
    expenditure_df["category"] = expenditure_df["merchant_name"].apply(map_category)
    expenditure_df["category"] = expenditure_df["category"].fillna("미분류")
    
    print(f"최종 매핑 완료: {len(expenditure_df)}건 (정산월: {statement_date})")
    display(expenditure_df.head(20))
else:
    expenditure_df = pd.DataFrame()

## 3. DB 저장

In [ ]:
if not expenditure_df.empty:
    insert_expenditure_data(expenditure_df)
    print(f"{len(expenditure_df)}개의 데이터가 성공적으로 저장되었습니다.")
else:
    print("저장할 데이터가 없습니다.")